# SOXX 官方 NAV 与冻结 Qlib 价格比例来源审计

只读同日来源比例、缺失及公司行动尺度，不读取未来收益或建立交易规则。


In [1]:
import os,sys,json,hashlib,re,datetime,math,tempfile,tarfile,subprocess,shlex
from pathlib import Path
from xml.etree import ElementTree as ET
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import archive_nas_recorder,_settings
card=root/'research/stage1_soxx_nav_price_alignment_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(root/src['official_NAV_raw'])==src['official_NAV_raw_sha256'] and sha(root/src['official_NAV_card']/'source_audit.json')==src['official_source_audit_sha256']
assert sha(root/src['Qlib_daily_price_provider']/'qlib.yaml')==src['Qlib_provider_config_sha256']
receipt=json.loads((root/src['official_NAV_card']/'nas_source_archive.json').read_text());assert receipt['verified'] and receipt['sha256']==src['official_NAS_archive_sha256']
assert all(plan['budget'][k]==0 for k in ('new_HTTP','new_model_fits','new_native_accounts','bootstraps'))
s=(root/src['official_NAV_raw']).read_text();s=re.sub(r'&(?!amp;|lt;|gt;|quot;|apos;|#[0-9]+;|#x[0-9A-Fa-f]+;)', '&amp;',s);xml=ET.fromstring(s);ns={'s':'urn:schemas-microsoft-com:office:spreadsheet'};key='{urn:schemas-microsoft-com:office:spreadsheet}Name';w=next(x for x in xml.findall('s:Worksheet',ns) if x.attrib.get(key)=='Historical')
nav={}
for row in w.findall('.//s:Row',ns)[1:]:
 vals=[c.findtext('s:Data',default='',namespaces=ns) for c in row.findall('s:Cell',ns)];d=pd.Timestamp(datetime.datetime.strptime(vals[0],'%b %d, %Y').date());v=float(vals[1]);assert d not in nav and math.isfinite(v) and v>0;nav[d]=v
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['Qlib_daily_price_provider']),region='us',kernels=1)
cal=pd.DatetimeIndex(D.calendar(start_time=plan['fixed_dates']['start'],end_time=plan['fixed_dates']['end'],freq='day'))
f=D.features(['SOXX'],['$close'],start_time=plan['fixed_dates']['start'],end_time=plan['fixed_dates']['end'],freq='day')
assert len(cal)==2205 and f.index.names==['instrument','datetime'] and len(f)==len(cal)
close=f.iloc[:,0].droplevel('instrument').rename('Qlib_close')
assert close.index.equals(cal) and np.isfinite(close).all() and (close>0).all()
nav_series=pd.Series(nav,name='issuer_NAV').reindex(cal)
assert nav_series.notna().all() and np.isfinite(nav_series).all() and (nav_series>0).all()
frame=pd.concat([close,nav_series],axis=1);frame['price_to_NAV']=frame['Qlib_close']/frame['issuer_NAV'];frame['log_price_to_NAV']=np.log(frame['price_to_NAV']);frame['adjacent_log_ratio_change']=frame['log_price_to_NAV'].diff()
assert np.isfinite(frame[['price_to_NAV','log_price_to_NAV']]).all().all()
summary_stats=lambda x:{'n':int(len(x)),'median':float(x.median()),'p05':float(x.quantile(.05)),'p95':float(x.quantile(.95)),'min':float(x.min()),'max':float(x.max())}
all_stats=summary_stats(frame['price_to_NAV']);year_stats={str(y):summary_stats(g['price_to_NAV']) for y,g in frame.groupby(frame.index.year)}
jumps=frame.loc[frame['adjacent_log_ratio_change'].abs()>0.1,['Qlib_close','issuer_NAV','price_to_NAV','adjacent_log_ratio_change']]
report={'decision':'read_only_NAV_Qlib_close_scale_audit_no_premium_feature','plan_sha256':sha(card/'plan.json'),'source_archive_verified':True,'Qlib_calendar_rows':len(cal),'matched_NAV_and_close_rows':len(frame),'missing_NAV_sessions':int(nav_series.isna().sum()),'overall_price_to_NAV':all_stats,'annual_price_to_NAV':year_stats,'adjacent_abs_log_ratio_change_gt_0p1_count':len(jumps),'largest_abs_adjacent_log_ratio_change':float(frame['adjacent_log_ratio_change'].abs().max()),'largest_abs_adjacent_date':str(frame['adjacent_log_ratio_change'].abs().idxmax().date()),'current_revision_not_PIT':True,'historical_publication_clock_unqualified':True,'new_model_fits':0,'new_native_accounts':0,'SOXX_future_outcomes_read':0}
static=pd.concat({'feature':frame},axis=1);static.index.name='datetime';static['instrument']='SOXX';static=static.set_index('instrument',append=True)
ds=DatasetH(handler=DataHandlerLP.from_df(static),segments={'source_overlap':(plan['fixed_dates']['start'],plan['fixed_dates']['end'])});assert len(ds.prepare('source_overlap',col_set=['feature']))==len(frame)
tracking=card/'qlib_recorder';assert not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);ds.config(dump_all=True,recursive=True)
 rec.save_objects(artifact_path='inputs',plan=plan,dataset=ds)
 rec.save_objects(artifact_path='analysis',source_frame=frame,ratio_jumps=jumps,report=report)
 rec.log_metrics(matched_sessions=len(frame),large_scale_jumps=len(jumps),new_model_fits=0,new_native_accounts=0)
(card/'result.json').write_text(json.dumps(report,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':report['plan_sha256'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'result':'result.json','new_model_fits':0,'new_native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Source alignment does not prove first publication time or old NAV versions.','Qlib close may be adjusted or normalized differently than issuer NAV; inspect ratio and corporate actions before treating it as premium.','No future label, information fit or full Stage1 account.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
e=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='soxx-nav-price-') as tmp:
 archive=Path(tmp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(e['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==e['sha256']
 with tarfile.open(archive,'r:gz') as z:assert any(m.name.endswith('/artifacts/analysis/report') for m in z.getmembers())
summary['nas_evidence']=e;summary['nas_readback']={'verified':True,'sha256':e['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps({'rows':len(frame),'ratio':all_stats,'years':{k:v['median'] for k,v in year_stats.items()},'jumps_gt_0p1':len(jumps),'largest_date':report['largest_abs_adjacent_date'],'recorder_id':rid},indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[22378:MainThread](2026-10-23 07:44:42,511) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[22378:MainThread](2026-10-23 07:44:42,513) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[22378:MainThread](2026-10-23 07:44:42,513) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


[22378:MainThread](2026-10-23 07:44:42,541) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


...

/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[22378:MainThread](2026-10-23 07:44:42,549) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name stage1_soxx_nav_price_alignment_20261023.


[22378:MainThread](2026-10-23 07:44:42,550) INFO - qlib.workflow - [exp.py:258] - Experiment 107693486411914435 starts running ...


[22378:MainThread](2026-10-23 07:44:42,607) INFO - qlib.workflow - [recorder.py:345] - Recorder 344717dcca444207994a80c089aa82a5 starts running under Experiment 107693486411914435 ...


[22378:MainThread](2026-10-23 07:44:43,656) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{
  "rows": 2205,
  "ratio": {
    "n": 2205,
    "median": 0.9675241860035173,
    "p05": 0.9244568393368809,
    "p95": 0.9993625627889156,
    "min": 0.9214215127333631,
    "max": 1.0017602564992272
  },
  "years": {
    "2018": 0.9269883380423849,
    "2019": 0.9392522714935736,
    "2020": 0.9520974665444396,
    "2021": 0.9607143566424137,
    "2022": 0.9681964591306724,
    "2023": 0.9800396900520214,
    "2024": 0.9876784888191273,
    "2025": 0.9950963284040768,
    "2026": 0.99945380282353
  },
  "jumps_gt_0p1": 0,
  "largest_date": "2020-03-18",
  "recorder_id": "344717dcca444207994a80c089aa82a5"
}


# SOXX 官方 NAV 与冻结 Qlib 收盘价：来源尺度审计

事先冻结官方 iShares NAV 原字节、Qlib `daily_prices` 提供器和 2018-01-02 至 2026-10-09 的完整重叠区间。原生 `D.calendar`、`D.features`、DatasetH 和 Recorder 得到 2,205 个同日 SOXX 收盘价与 NAV 对，零缺失。官方历史表的周末重复行仅通过 Qlib 交易日历排除；没有填充价格或读取未来收益。

两套数值**不能直接相除当作 ETF 溢折价**。`Qlib close / 官方 NAV` 的年度中位数从 2018 年 **0.92699** 平滑上升至 2026 年 **0.99945**；全期中位数 **0.96752**，5%–95% 分位为 **0.92446–0.99936**。相邻日对数比例变化超过 0.1 的日期为 0，最大绝对变化 0.01713（2020-03-18）。这说明当前两个历史序列存在持续变化的价格尺度或调整口径；本卡没有证据将其精确归因于哪一种分红、分拆或修订处理，也不能事后按年份归一化来制造溢折价。

Qlib Recorder `344717dcca444207994a80c089aa82a5` 的报告、2,205 行 DatasetH 与原件哈希已读回；Recorder 和官方原件分别归档 Yanxuan NAS 并按哈希冷读。0 新模型拟合、0 新账户。**决定：关闭“官方 NAV 直接拼冻结 Qlib close”这个固定来源版本；不建因子、不推广。** 若继续研究 ETF 溢折价，需先取得与官方 NAV 同拆股/分红口径的原始历史市场收盘价，验证两者当时的发布时间及版本，再冻结信息检验。完整 Stage1 0bp 超额 SOXX 目标仍未达成。
